# Pset 2 Problem 2: Sentence acceptability, probability, and language models

In this problem, you will evaluate two very different language models. While both models share the same basic task of assigning probabilities to strings, they have different implementations which can give rise to differing behavior.

## Getting started

**First**, make a copy of this notebook so you can make your own changes. Click *File -> Save a copy in Drive*.

### What you need to do

Read through this notebook and execute each cell in sequence, making modifications and adding code where necessary. You should execute all of the code as instructed, and make sure to write code or textual responses wherever the text **TODO** shows up in text and code cells.

When you're finished, choose *File -> Download .ipynb*. You will upload this `.ipynb` file as part of your submission.

## 0. Acceptability Data

In this section we load and merge 3 datasets compiling crowd-sourced acceptability judgments gathered in 3 corresponding large-scale experiments ([Lau et al., 2017](https://onlinelibrary.wiley.com/doi/full/10.1111/cogs.12414)). The `adger_filtered` dataset uses sentences from a linguistics textbook ([Adger, 2003](http://www.pnu.ac.ir/Portal/file/?985502/nahv.pdf%20)). It contains grammatical and ungrammatical sentences. The `bnc` dataset uses sentences from the British National Corpus. The `enwiki` dataset uses sentences from English Wikipedia. In addition to the raw English sentences (labeled `en`), the `bnc` and `enwiki` corpora involve grammatically "noisy" counterparts obtained by translating the raw sentences to another language (Norwegian, Spanish, Chinese, or Japanese), and then translating the sentences back to English. Those sentences, which are expected to have intermediary grammaticality scores, are labeled `no`, `es`, `zh` and `ja` respectively.

In [ ]:
import pandas as pd
import random
import numpy as np
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [ ]:
# mount Google drive to persist files when kernel is restarted
from google.colab import drive
drive.mount('/content/gdrive')
GDRIVE_DIR = "/content/gdrive/My Drive/cogai-pset-2"

In [ ]:
# This code chunk needs to be run only the first time through the pset.
!wget https://gu-clasp.github.io/projects/smog/data/adger_filtered.csv
!wget https://gu-clasp.github.io/projects/smog/data/bnc.csv
!wget https://gu-clasp.github.io/projects/smog/data/enwiki.csv

!mkdir -p "$GDRIVE_DIR"
!mv adger_filtered.csv "$GDRIVE_DIR/"
!mv bnc.csv "$GDRIVE_DIR/"
!mv enwiki.csv "$GDRIVE_DIR/"

In [ ]:
# this function parses the 3 datasets, extracts and puts together the relevant data.
def merge_datasets():
    sentences = []
    ratings = []
    acceptabilities = []
    datasets = []
    # we iterate over the 3 datasets...
    for fname in ['adger_filtered', 'bnc', 'enwiki']:
        df = pd.read_csv(f'{GDRIVE_DIR}/{fname}.csv', sep='\t')
        ss = df['text'].values.tolist() #the sentences
        sentences.extend(ss)
        rs = df['mean_rating'].values.tolist() #the sentences' mean human ratings
        mops = df['MOP'].values.tolist() #parameters describing the ratings

        ''' the following line converts the mean human ratings that were on a
        1-100 scale to a 1-4 scale, to match the other ratings.'''
        rs = [r/33+32/33 if mop == 'MOP100' else r for (r, mop) in zip(rs, mops)]
        ratings.extend(rs)

        if fname == 'adger_filtered':
            accs = df['language'].values.tolist()
            # y is for grammatical english sentences, n for ungrammatical ones
            accs = ['y' if a == 'adger-good' else 'n' for a in accs ]
        else:
            accs = df['language'].values.tolist()
        acceptabilities.extend(accs)
        datasets.extend([fname]*len(rs))


    df = pd.DataFrame()
    df['original_dataset'] = datasets
    df['sentence'] = sentences
    df['grammatical?'] = acceptabilities
    df['mean_human_rating'] = ratings
    return df

In [ ]:
df = merge_datasets()

Let's look at a sample of the dataset...

In [ ]:
df.head()

In [ ]:
df_sample = df.sample(n=5, random_state=42)
for i in df_sample.index:
    print(f"Sentence: {df_sample['sentence'][i]}")
    print(f"Original dataset: {df_sample['original_dataset'][i]}")
    print(f"Theoretical grammaticality score (y/n for pure english sentence, language code for 'noisy' sentences): {df_sample['grammatical?'][i]}")
    print(f"Mean human grammaticality score {df_sample['mean_human_rating'][i]}\n")

A sanity check to see if the distributions of mean human ratings make sense...

In [ ]:
sns.violinplot(
    data=df,
    x="grammatical?",
    y="mean_human_rating",
    order=["y", "en", "no", "es", "zh", "ja", "n"]
)

'''Codes for sentence type:
   y: grammatical sentence from the Adger textbook
   en: grammatical sentence from BNC/Wiki
   no: English->Norwegian->English backtranslated
   es: English->Spanish->English backtranslated
   zh: English->Chinese->English backtranslated
   ja: English->Japanese->English backtranslated
   n: ungrammatical sentence from the Adger textbook
'''
plt.xlabel('Sentence type')
plt.ylabel('Mean human ratings (1->4)')
plt.show()

We are using a "violin plot", which is similar to a box plot but shows the shape of the distribution rather than simply the summary statistics. It looks like the human rating follow reasonable expectations: the grammatical sentences are found to be most acceptable, the ungrammatical sentences are least acceptable, and the noisy "round-trip translation" sentences are somewhere in between, with intermediate languages that are more similar to English leading to higher ratings than languages that are more distant (observe the sharp drop between Indo-European and non Indo-European languages!).

## 1. N-gram Language Models

N-gram language models used a fixed context window to estimate word probabilities based on preceding context. As discussed in class, there are a variety of smoothing methods to address the problem of data sparsity (i.e. estimating word probabilities when the preceding n-gram context has not been observed during training).

###Loading the model

In [ ]:
# Install the required packages

!pip install kenlm
!pip install sentencepiece
!pip install huggingface-hub

In [ ]:
# Download the trained n-gram model parameters and vocabulary
# Note that the pre-trained n-gram model is about 4GB and may take a couple of
# minutes to download.
# This code chunk needs to be run only the first time through the pset.
!wget "https://huggingface.co/edugp/kenlm/resolve/main/wikipedia/en.arpa.bin"
!wget "https://huggingface.co/edugp/kenlm/resolve/main/wikipedia/en.sp.model"
!wget "https://huggingface.co/edugp/kenlm/resolve/main/wikipedia/en.sp.vocab"

!mv en.arpa.bin "$GDRIVE_DIR/"
!mv en.sp.model "$GDRIVE_DIR/"
!mv en.sp.vocab "$GDRIVE_DIR/"

#### Technical Detail: Tokenization

In language modeling, a string is broken up into _tokens_, which are the vocabulary items that the model is predicting a distribution over, given some context. Tokens very roughly correspond to our notion of a "word", but there are different approaches for turning a string into a list of tokens, a process known as _tokenization_.

Most modern language modeling approaches break strings into sub-word tokens from a fixed vocabulary. For example, there could be 50,000 tokens in the vocabulary, and any string of characters could be broken up into one or more of these tokens. Highly frequent words, such as "a" or "the", could have their own token. Longer or more complex words, like "antimicrobial", might be broken up into multiple tokens. The advantage of this approach is that this system of sub-word tokens can handle previously unseen words within a fixed vocabulary by breaking the unseen words into smaller pieces.

### Evaluation metrics

The code below defines a n-gram language model, which is initialized using pre-trained parameters and provides functionality for estimating the token-by-token probabilities of linguistic strings based on context.

You do not need to understand every aspect of the code, some of which handles lower-level details such as normalizing capitalization and accent marks.

In [ ]:
import os
import re
import unicodedata
from typing import Dict
import kenlm
import sentencepiece
# from huggingface_hub import cached_download, hf_hub_url


class SentencePiece:
    """
    SentencePiece is a tokenization method that breaks up strings into a
    vocabulary of a pre-defined size consisting of tokens (which do not
    necessarily correspond to the typical linguistic notion of words)
    """
    def __init__(
        self, model: str,
    ):
        super().__init__()
        self.sp = sentencepiece.SentencePieceProcessor()
        self.sp.load(str(model))

    def do(self, text: dict) -> dict:
        tokenized = self.sp.encode_as_pieces(text)
        return " ".join(tokenized)


class KenlmModel:
    digit_re: re.Pattern = re.compile(r"\d")
    a = None
    sentence_piece_model_dir = None

    def __init__(
        self,
        lower_case: bool = False,
        remove_accents: bool = False,
        normalize_numbers: bool = True,
        punctuation: int = 1,
    ):
        self.model = kenlm.Model(os.path.join(GDRIVE_DIR, "en.arpa.bin"))
        self.tokenizer = SentencePiece(os.path.join(GDRIVE_DIR, "en.sp.model"))
        self.accent = remove_accents
        self.case = lower_case
        self.numbers = normalize_numbers
        self.punct = punctuation

    @classmethod
    def from_pretrained(cls,):
        return cls(False, False, True, 1,)

    def per_word_logprobs(self, sent: str):
        log_probs = []
        contexts = []
        lengths = []
        sent = self.normalize(
            sent,
            accent=self.accent,
            case=self.case,
            numbers=self.numbers,
            punct=self.punct
        )
        sentence_tkn = self.tokenizer.do(sent)
        tokens = ["<BOS>"] + sentence_tkn.split() + ["<EOS>"]
        for (i, (prob, length, oov)) in enumerate(
            self.model.full_scores(sentence_tkn, bos=True, eos=True)
        ):
            assert (
                not oov
            ), "Sentence contained an out-of-vocabulary word -- this should not happen"
            log_probs.append(prob)
            contexts.append(tokens[i - length + 2 : i + 1])
            lengths.append(length)
        return pd.DataFrame(
            {
                "token": tokens[1:],
                "log_prob": log_probs,
                "ngram_order": lengths,
                "context": contexts,
            }
        )

    def is_sent_computable(self, sent: str):
        sent = self.normalize(
            sent,
            accent=self.accent,
            case=self.case,
            numbers=self.numbers,
            punct=self.punct
        )
        sentence_tkn = self.tokenizer.do(sent)
        oovs = [t[2] for t in self.model.full_scores(sentence_tkn, bos=True, eos=True)]
        if True in oovs:
            return False
        return True

    def normalize(
        self,
        line: str,
        accent: bool = True,
        case: bool = True,
        numbers: bool = True,
        punct: int = 1,
    ) -> str:
        line = line.strip()
        if not line:
            return line
        if case:
            line = line.lower()
        if accent:
            line = self.strip_accents(line)
        if numbers:
            line = self.digit_re.sub("0", line)
        return line

    def strip_accents(self, line: str) -> str:
        """Strips accents from a piece of text."""
        nfd = unicodedata.normalize("NFD", line)
        output = [c for c in nfd if unicodedata.category(c) != "Mn"]
        if len(output) == line:
            return line
        return "".join(output)

    def sent_logprob(self, sent: str):
        # TODO: implement this function
        return np.nan

    def sent_normalized_logprob(self, sent: str):
        # TODO: implement this function
        return np.nan


In [ ]:
# we load the n-gram model
ngram_model = KenlmModel.from_pretrained()

Note that because of the SentencePiece tokenizer, even strings of non-words should have computable probabilities if they can be decomposed into valid tokens. Note that in the tokenized version of the string, the special underscore symbol "_" indicates that a token is starting a new word; this makes it possible to reaggregate the original words.

In [ ]:
s = 'spronglet gyrtam thusker'
print(ngram_model.tokenizer.do(s))
print(ngram_model.is_sent_computable(s))

The `per_word_logprobs()` function in the KenlmModel class defined above takes a string and returns a dataframe containing per-token log probabilities for each word in context. The model uses modified Kneser-Ney smoothing, and so while the model has a maximum n-gram length, it can use smaller amounts of context to estimate probabilities for sparser contexts.

See http://www.foldl.me/2014/kneser-ney-smoothing/ for more information.

The function returns the results as a Pandas dataframe, with the following columns:
- token: the token at the given position in the sentence
- log_prob: the log probability assigned by the model
- ngram_order: the length of the longest n-gram used to calculate the probability according to the Kneser-Ney model
- context: the context used to calculate the probability according to the Kneser-Ney model

In [ ]:
# From kenlm example: https://github.com/kpu/kenlm/blob/master/python/example.py#L20
sent = "I bought a gallon of water."
ngram_results = ngram_model.per_word_logprobs(sent)
ngram_results

**TODO:** Your task is to fill in the `sent_logprob()` and `sent_normalized_logprob()` functions in the KenlmModel class above.

`sent_logprob()` should return the summed per-token logprobs across a sentence.

`sent_normalized_logprob()` should return the average per-token logprob across a sentence. Intuitively, this accounts for differences in sentence length and provides a measure of how predictable the average token in a sentence is.

Confirm that your functions work by running the cell below and seeing if the numbers you get for the different sentences are reasonable.

In [ ]:
sentences = [
    "I bought gas",
    "I bought tar",
    "I bought a gallon of gas",
    "I bought a gallon of tar",
]

for sent in sentences:
  print("Sentence:", sent)
  print("Summed Log Probability:", ngram_model.sent_logprob(sent))
  print("Mean Log Probability (per token):", ngram_model.sent_normalized_logprob(sent), "\n")

###Comparing the models' predictions to human ratings

To compare the model-generated probabilities to human acceptability ratings, we will employ both qualitative and quantitative comparisons:
- Visualize the correlation between model scores and human ratings
- Compute the Spearman correlation coefficient between model scores and human ratings

In [ ]:
df = merge_datasets()

# filter sentences from the dataset that have out-of-vocabulary items (usually weird punctuation)
len_orig = len(df)
df = df[df.apply(lambda r: ngram_model.is_sent_computable(r['sentence']), axis=1)]
len_filter = len(df)

print(f"Dropped {len_orig - len_filter} sentences due to OOV items")

We will now apply the functions you wrote to each sentence in the dataset. The code below adds a new column in the dataframe corresponding to the summed sentence log probability of each sentence.

In [ ]:
df["sentence_log_prob_ngram"] = df.sentence.apply(lambda x: ngram_model.sent_logprob(x))
df

Let's take a look at the distribution of n-gram scores for each of the corpora in our merged dataset.

In [ ]:
sns.violinplot(
    data=df,
    x="grammatical?",
    y="sentence_log_prob_ngram",
    order=["y", "en", "no", "es", "zh", "ja", "n"]
)
plt.title("N-gram Model Score (unnormalized)")
plt.xlabel("sentence type")
plt.show()

Let's view the correlation between the n-gram scores and the human ratings. The code below will show a plot with 3 facets, one for each of the 3 original datasets. The scatterplot will also be overlaid with a regression line showing the direction of correlation (assuming a linear relationship). The width of the confidence intervals around the regression line can give you a qualitative sense of how reliable the correlation is.

In [ ]:
g = sns.FacetGrid(data=df, col="original_dataset")
g.map_dataframe(
    sns.regplot,
    x="sentence_log_prob_ngram",
    y="mean_human_rating",
    scatter_kws={"alpha": 0.1}
)

In [ ]:
for dataset in ['adger_filtered', 'bnc', 'enwiki']:
  df_sub = df.query("original_dataset == @dataset")
  print(f"{dataset}: ", scipy.stats.spearmanr(df_sub.mean_human_rating,
                                              df_sub.sentence_log_prob_ngram))

print("Overall correlation: ",
      scipy.stats.spearmanr(df.mean_human_rating,
                            df.sentence_log_prob_ngram))

Now we repeat the process for the normalized version of the n-gram probability score. A new column appears, and its values should correspond to the existing `sentence_log_prob_ngram` column, but divided by the number of tokens in the sentence.

In [ ]:
df["sentence_normalized_log_prob_ngram"] = df.sentence.apply(
    lambda x: ngram_model.sent_normalized_logprob(x)
)
df

N-gram scores depending on sentence type:

In [ ]:
sns.violinplot(
    data=df,
    x="grammatical?",
    y="sentence_normalized_log_prob_ngram",
    order=["y", "en", "no", "es", "zh", "ja", "n"]
)
plt.title("N-gram Model Score (normalized)")
plt.xlabel("sentence type")
plt.show()

Correlation with human ratings depending on the dataset:

In [ ]:
g = sns.FacetGrid(data=df, col="original_dataset")
g.map_dataframe(
    sns.regplot,
    x="sentence_normalized_log_prob_ngram",
    y="mean_human_rating",
    scatter_kws={"alpha": 0.1}
)

In [ ]:
for dataset in ['adger_filtered', 'bnc', 'enwiki']:
  df_sub = df.query("original_dataset == @dataset")
  print(f"{dataset}: ", scipy.stats.spearmanr(df_sub.mean_human_rating,
                                              df_sub.sentence_normalized_log_prob_ngram))

print("Overall correlation: ",
      scipy.stats.spearmanr(df.mean_human_rating,
                            df.sentence_normalized_log_prob_ngram))

Correlation with human ratings depending sentence type:

In [ ]:
g = sns.FacetGrid(data=df, col="grammatical?", col_wrap=3, height=2)
g.map_dataframe(
    sns.regplot,
    x="sentence_normalized_log_prob_ngram",
    y="mean_human_rating",
    scatter_kws={"alpha": 0.1}
)

##Neural Models (Large Language Models)

While we have mainly discussed n-gram language models in class, you have probably heard of large language models (LLMs), which are neural networks trained on large amounts of data which have excelled at predicting words in context. For this next section, we will investigate whether LLMs more closely match human acceptability judgments, compared to n-gram models.

Based on: https://github.com/MuhammadMoinFaisal/LargeLanguageModelsProjects/blob/main/Fine-Tune%20Llama%202/Fine_tune_Llama_2_in_Google_Colab.ipynb
AND notebooks by Mycal Tucker





In [ ]:
%%capture
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

###Loading the model

The following code loads the neural language model and its associated tokenizer.

In [ ]:
# The model that you want to train from the Hugging Face hub
model_name = "gpt2-medium"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
# device_map = {"": "cuda:0"}

# Load base model
neural_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # device_map=device_map,
    output_hidden_states=True
)
neural_model.config.use_cache = False
neural_model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
neural_model = neural_model.to(device)

Note that the tokenizer for this model is different than the tokenizer used by the n-gram model, meaning the same sentence could be split into different tokens according to the two methods. Note the special "Ġ" symbol which is prepended to tokens to indicate that it is starting a new word -- this makes it possible to re-aggregate sub-word tokens into their original words (e.g. "Ġantim" +  "icrobial" = "antimicrobial").

In [ ]:
encoded = tokenizer.encode("They use antimicrobial soap")
print(encoded)
print(tokenizer.convert_ids_to_tokens(encoded))

**Optional:** For the purpose of this assignment, we are using `gpt2` as our large language model. However, gpt2 is now far from the state-of-the-art LLM, so you are welcome to experiment with a more recent model, LlaMa-2. Due to the size of the model, some slightly hacky workarounds are required to get this to work on Colab. Some code below is included to get you started on using LlaMa, but it is not guaranteed to work within the computational constraints of Colab.

In [ ]:
# Advanced users: Uncomment the following code in case you are using 4bit quantization to load a very large model

# model_name = "NousResearch/Llama-2-7b-hf"

# # Activate 4-bit precision base model loading
# use_4bit = True

# # Compute dtype for 4-bit base models
# bnb_4bit_compute_dtype = "float16"

# # Quantization type (fp4 or nf4)
# bnb_4bit_quant_type = "nf4"

# # Activate nested quantization for 4-bit base models (double quantization)
# use_nested_quant = False

# # Load the entire model on the GPU 0
# device_map = {"": 0}

# compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=use_4bit,
#     bnb_4bit_quant_type=bnb_4bit_quant_type,
#     bnb_4bit_compute_dtype=compute_dtype,
#     bnb_4bit_use_double_quant=use_nested_quant,
# )

# # Check GPU compatibility with bfloat16
# if torch.cuda.is_available() and compute_dtype == torch.float16 and use_4bit:
#     major, _ = torch.cuda.get_device_capability()
#     if major >= 8:
#         print("=" * 80)
#         print("Your GPU supports bfloat16: accelerate training with bf16=True")
#         print("=" * 80)

# # Load base model
# llama_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map=device_map,
#     output_hidden_states=True
# )
# llama_model.config.use_cache = False
# llama_model.config.pretraining_tp = 1

# # Load LLaMA tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

###Evaluation metrics

Training and evaluation of LLMs usually involves a Graphics Processing Unit (GPU) to accelerate the matrix operations involved in neural networks. A GPU can perform inference on multiple input sentences (a 'batch') simultaneously. To take advantage of this, the code provided below assumes that inference is run on a list of sentences rather than a single sentence. Note the differences between this code and the n-gram model code from before.

In [ ]:
# a function that takes a list of sentences and returns a list of dataframes,
# each containing an alignment of tokens and their contextual log probabilities
def per_word_logprobs(list_of_sentences):
    # tokenize (include adding a special token which indicates beginning of new text)
    df = pd.DataFrame(list_of_sentences)
    df[0] = "<|endoftext|>" + df[0]
    tokenized = df[0].apply(lambda x: tokenizer.encode(x))

    # handle padding and attention mask
    max_len = 0
    for i in tokenized.values:
      if len(i) > max_len:
          max_len = len(i)
    tokenized = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])
    attention_mask = np.where(tokenized != 0, 1, 0)
    attention_mask = torch.tensor(attention_mask).to(device)

    # feed input ids to model
    input_ids = torch.tensor(tokenized).to(device)
    with torch.no_grad():
        model_output = neural_model(input_ids, attention_mask=attention_mask)

        # logits are the raw outputs of the model, i.e. activation levels
        # for each item in the vocabulary. we convert logits to probabilities
        # with the softmax function and apply a log transform (base 10 for)
        # consistency with n-gram model implementation
        logits = model_output.get('logits').detach().cpu()
        logprobs = logits.softmax(axis=2).log10()

    start_idx = 1

    results = []
    for i, ids in enumerate(input_ids.cpu()):
        # get the last non-masked idx and only extract log_probs up to that idx
        end_idx = (ids != 0).nonzero().flatten()[-1].item()
        sent_logprobs = logprobs[i, range(end_idx), ids[start_idx:end_idx + 1]].numpy()
        sent_toks = tokenizer.convert_ids_to_tokens(ids[start_idx:end_idx + 1])
        results.append(pd.DataFrame({
            "token": sent_toks,
            "log_prob": sent_logprobs
        }))
    return results

def sent_logprob(list_of_sentences):
    # TODO: implement this function
    return [np.nan] * len(list_of_sentences)

def sent_normalized_logprob(list_of_sentences):
    # TODO: implement this function
    return [np.nan] * len(list_of_sentences)


**TODO:** Similarly to above, your task is to fill in the `sent_logprob()` and `sent_normalized_logprob()` functions in the cell above for use with the neural LM.

- `sent_logprob()` should return the summed per-token logprobs across a sentence.

- `sent_normalized_logprob()` should return the average per-token logprob across a sentence.

Confirm that your functions work by running the cell below and seeing if the numbers you get for the different sentences are reasonable.

In [ ]:
sentences = ["I bought a gallon of milk",
             "I bought a gallon of soap",
             "Modernization and industrialization of the nineteenth century Britain changed population map."]
logprobs = per_word_logprobs(sentences)
display(logprobs[0])
display(logprobs[1])
display(logprobs[2])
print(sent_logprob(sentences))
print(sent_normalized_logprob(sentences))

###Comparing the models' predictions to human ratings

The code below computes scores for the sentences in the dataset using the `sent_logprob()` function that you wrote. Note that the code splits the sentences into batches before feeding them into the model.

In [ ]:
from tqdm import tqdm

batch_size = 32
log_probs_neural = []
for batch in tqdm(np.array_split(df, len(df) // batch_size)):
  log_probs_neural.extend(sent_logprob(batch.sentence.tolist()))

In [ ]:
df["sentence_log_prob_neural"] = log_probs_neural

# NOTE: we can actually just avoid redundant computation by calculating the
# normalized values from the unnormalized, without passing through the model again
df["token_count"] = df.sentence.apply(lambda x: len(tokenizer.encode(x)))
df["sentence_normalized_log_prob_neural"] = df.sentence_log_prob_neural / df.token_count

Let's take a look at the sentences which had the _lowest_ probability under the LLM. The reason for the sentences having such low probability under the model should be fairly interpretable.

In [ ]:
df.nsmallest(10, "sentence_log_prob_neural").sentence.tolist()

In [ ]:
sns.violinplot(
    data=df,
    x="grammatical?",
    y="sentence_log_prob_neural",
    order=["y", "en", "no", "es", "zh", "ja", "n"]
)
plt.title("Neural LM Score (unnormalized)")
plt.xlabel("sentence type")
plt.show()

Correlation with human ratings depending on the dataset:

In [ ]:
g = sns.FacetGrid(data=df, col="original_dataset")
g.map_dataframe(
    sns.regplot,
    x="sentence_log_prob_neural",
    y="mean_human_rating",
    scatter_kws={"alpha": 0.1}
)

In [ ]:
for dataset in ['adger_filtered', 'bnc', 'enwiki']:
  df_sub = df.query("original_dataset == @dataset")
  print(f"{dataset}: ", scipy.stats.spearmanr(df_sub.mean_human_rating,
                                              df_sub.sentence_log_prob_neural))

print("Overall correlation: ",
      scipy.stats.spearmanr(df.mean_human_rating,
                            df.sentence_log_prob_neural))

In [ ]:
sns.violinplot(
    data=df,
    x="grammatical?",
    y="sentence_normalized_log_prob_neural",
    order=["y", "en", "no", "es", "zh", "ja", "n"]
)
plt.title("Neural LM Score (normalized)")
plt.xlabel("sentence type")
plt.show()

Correlation with human ratings depending on the dataset:

In [ ]:
g = sns.FacetGrid(data=df, col="original_dataset")
g.map_dataframe(
    sns.regplot,
    x="sentence_normalized_log_prob_neural",
    y="mean_human_rating",
    scatter_kws={"alpha": 0.1}
)

In [ ]:
for dataset in ['adger_filtered', 'bnc', 'enwiki']:
  df_sub = df.query("original_dataset == @dataset")
  print(f"{dataset}: ", scipy.stats.spearmanr(df_sub.mean_human_rating,
                                              df_sub.sentence_normalized_log_prob_neural))

print("Overall correlation: ",
      scipy.stats.spearmanr(df.mean_human_rating,
                            df.sentence_normalized_log_prob_neural))

## 3. Implementing the SLOR acceptability measure

Read Section 3.2 of [Lau et al., 2017](https://onlinelibrary.wiley.com/doi/full/10.1111/cogs.12414). They mention a metric called SLOR to use as an acceptability measure for sentences based on language model probabilities and unigram probabilities (i.e. baseline word frequencies in the language).

In this section, you will implement the SLOR metric and create figures comparing it to human ratings.

The SLOR metric relies on unigram probabilities. The `en.sp.vocab` file actually contains log unigram probabilities for each of the SentencePiece tokens in the n-gram model's vocabulary. You will use these to help calculate SLOR.

In [ ]:
# loading unigram probabilities from the file into a dictionary
unigram_logprobs = {}
with open(os.path.join(GDRIVE_DIR, "en.sp.vocab")) as f:
  for i, line in enumerate(f):
    word, logprob = line.split()
    unigram_logprobs[word] = float(logprob)

# print the top words in the vocabulary
list(unigram_logprobs.items())[:10]

In [ ]:
# TODO: a function that computes the summed unigram log probability
# for a given sentence. Note that we only have access to the unigram
# probabilities according to the KenLM model's SentencePiece
# tokenizer, so make sure you first encode the sentence using this
# tokenizer before performing the lookup of the unigram log-probs
def sent_log_prob_unigram(sentence):
  return np.nan

In [ ]:
# Adding a column to the dataframe corresponding to the SLOR score according to n-gram probabilities
df["sentence_log_prob_unigram"] = df.sentence.apply(sent_log_prob_unigram)
df["token_count_sentencepiece"] = df.sentence.apply(lambda x: len(ngram_model.tokenizer.do(x).split()))
df["slor_ngram"] = (df.sentence_log_prob_ngram - df.sentence_log_prob_unigram) / df.token_count_sentencepiece

In [ ]:
g = sns.FacetGrid(data=df, col="original_dataset")
g.map_dataframe(
    sns.regplot,
    x="slor_ngram",
    y="mean_human_rating",
    scatter_kws={"alpha": 0.1}
)

In [ ]:
for dataset in ['adger_filtered', 'bnc', 'enwiki']:
  df_sub = df.query("original_dataset == @dataset")
  print(f"{dataset}: ", scipy.stats.spearmanr(df_sub.mean_human_rating,
                                              df_sub.slor_ngram))

print("Overall correlation: ",
      scipy.stats.spearmanr(df.mean_human_rating,
                            df.slor_ngram))

In [ ]:
# Adding a column to the dataframe corresponding to the SLOR score according to LLM probabilities
df["slor_neural"] = (df.sentence_log_prob_neural - df.sentence_log_prob_unigram) / df.token_count_sentencepiece

In [ ]:
g = sns.FacetGrid(data=df, col="original_dataset")
g.map_dataframe(
    sns.regplot,
    x="slor_neural",
    y="mean_human_rating",
    scatter_kws={"alpha": 0.1}
)

In [ ]:
for dataset in ['adger_filtered', 'bnc', 'enwiki']:
  df_sub = df.query("original_dataset == @dataset")
  print(f"{dataset}: ", scipy.stats.spearmanr(df_sub.mean_human_rating,
                                              df_sub.slor_neural))

print("Overall correlation: ",
      scipy.stats.spearmanr(df.mean_human_rating,
                            df.slor_neural))

## Questions

**TODO:** Looking at the various plots, which metric seems to best capture human ratings and why? Do you identify any interesting difference of behavior between the n-gram model and the neural model?

 **TODO:** Design your own sentence(s) where there is a dramatic difference in conditional word probabilities for the same (word,context) pair for the fitted n-gram model versus the neural model, and explain the reason(s) for the differences.

In [ ]:
sent = "Your sentence goes here."
result = ngram_model.per_word_logprobs(sent)
result

In [ ]:
per_word_logprobs([sent])[0]